# Pilot CLIP Embeddings

## Setup

Use the existing Colab environment and validated Drive images. No package installation or image copies.

In [43]:
from collections import Counter
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path, PurePosixPath
import hashlib
import json
import platform
import tempfile
import time

try:
    import numpy as np
    import pandas as pd
    import torch
    import open_clip
    import pyarrow
    from PIL import Image
except ImportError as error:
    raise RuntimeError(f"STOP: required dependency unavailable: {error}") from error

In [44]:
repository_root = Path(
    "/content/Out-of-Context-Multimodal-Misinformation-Detection-Image-Context-Verification"
)
drive_root = Path("/content/drive/MyDrive/MajorProjectAssets")
image_root = drive_root / "pilot_images"
embedding_root = drive_root / "embeddings/pilot"
manifest_root = repository_root / "data/manifests"
model_name = "ViT-B-32"
pretrained = "openai"
batch_size = 64
diagnostic_seed = 42

assert repository_root.is_dir(), "STOP: repository directory unavailable"
assert image_root.is_dir(), "STOP: canonical Drive image directory unavailable"
assert drive_root.is_dir(), "STOP: mounted Drive unavailable"
embedding_root.mkdir(parents=True, exist_ok=True)
cuda_available = torch.cuda.is_available()
device = torch.device("cuda" if cuda_available else "cpu")
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None

In [45]:
environment = {
    "python_version": platform.python_version(),
    "pytorch_version": torch.__version__,
    "openclip_version": version("open_clip_torch"),
    "cuda_available": cuda_available,
    "cuda_version": torch.version.cuda,
    "gpu_name": gpu_name,
    "device": str(device),
    "repository_root": str(repository_root),
    "image_root": str(image_root),
    "embedding_root": str(embedding_root),
}
display(pd.Series(environment, name="value").to_frame())

,value
python_version,3.12.13
pytorch_version,2.11.0+cu128
openclip_version,3.3.0
cuda_available,True
cuda_version,12.8
gpu_name,Tesla T4
device,cuda
repository_root,/content/Out-of-Context-Multimodal-Misinformat...
image_root,/content/drive/MyDrive/MajorProjectAssets/pilo...
embedding_root,/content/drive/MyDrive/MajorProjectAssets/embe...


## Manifests

In [46]:
manifest_files = {
    "queries": manifest_root / "pilot_queries.parquet",
    "evidence": manifest_root / "pilot_evidence.parquet",
    "assets": manifest_root / "pilot_image_assets.parquet",
}
for name, manifest_file in manifest_files.items():
    assert manifest_file.is_file(), f"STOP: {name} manifest unavailable: {manifest_file}"

manifest_hashes = {}
for name, manifest_file in manifest_files.items():
    manifest_hashes[name] = hashlib.sha256(manifest_file.read_bytes()).hexdigest()

queries = pd.read_parquet(manifest_files["queries"])
evidence = pd.read_parquet(manifest_files["evidence"])
assets = pd.read_parquet(manifest_files["assets"])

In [47]:
def check_manifest(name, table, expected_rows, required_columns):
    assert len(table) == expected_rows, f"STOP: unexpected {name} row count"
    missing_columns = set(required_columns) - set(table.columns)
    assert not missing_columns, f"STOP: {name} missing columns: {sorted(missing_columns)}"
    for column in required_columns:
        assert not table[column].isna().any(), f"STOP: {name}.{column} contains nulls"

check_manifest("queries", queries, 200, [
    "query_id", "image_id", "query_image_path", "submitted_caption",
])
check_manifest("evidence", evidence, 5000, ["id", "image_path", "source", "caption"])
check_manifest("assets", assets, 5000, [
    "image_path", "is_query_image", "is_evidence_image", "asset_role", "source", "metadata_id",
])
assert queries["query_id"].is_unique
assert queries["query_image_path"].is_unique
assert evidence["image_path"].is_unique
assert evidence["id"].is_unique
assert assets["image_path"].is_unique
for captions in (evidence["caption"], queries["submitted_caption"]):
    for caption in captions:
        assert isinstance(caption, str) and caption.strip(), "STOP: invalid caption"

In [48]:
query_paths = set(queries["query_image_path"])
evidence_paths = set(evidence["image_path"])
asset_paths = set(assets["image_path"])
assert evidence_paths == asset_paths
assert query_paths.issubset(evidence_paths)
assert set(assets.loc[assets["is_query_image"], "image_path"]) == query_paths
assert set(assets.loc[assets["is_evidence_image"], "image_path"]) == evidence_paths

asset_lookup = assets.set_index("image_path")
for row in evidence.itertuples(index=False):
    asset = asset_lookup.loc[row.image_path]
    assert asset["metadata_id"] == row.id
    assert asset["source"] == row.source

display(pd.DataFrame({
    "manifest": ["queries", "evidence", "assets"],
    "rows": [len(queries), len(evidence), len(assets)],
    "unique_image_paths": [len(query_paths), len(evidence_paths), len(asset_paths)],
}))

,manifest,rows,unique_image_paths
0,queries,200,200
1,evidence,5000,5000
2,assets,5000,5000


In [49]:
def resolve_image_path(image_path):
    assert isinstance(image_path, str) and image_path, "STOP: empty image path"
    parts = image_path.split("/")
    relative_path = PurePosixPath(image_path)
    assert not relative_path.is_absolute(), f"STOP: absolute image path: {image_path}"
    assert "\\" not in image_path, f"STOP: invalid separator: {image_path}"
    assert not any(part in ("", ".", "..") for part in parts), f"STOP: unsafe path: {image_path}"
    assert len(parts) == 6 and parts[:2] == ["visual_news", "origin"]
    assert parts[3] == "images" and relative_path.suffix == ".jpg"
    return image_root.joinpath(*parts)

availability_started = time.perf_counter()
missing_paths = []
for image_path in evidence["image_path"]:
    if not resolve_image_path(image_path).is_file():
        missing_paths.append(image_path)
availability_seconds = time.perf_counter() - availability_started
print(f"Selected files found: {len(evidence) - len(missing_paths)}/{len(evidence)}")
print(f"Availability check: {availability_seconds:.1f} seconds")
assert not missing_paths, f"STOP: missing Drive images: {missing_paths[:10]}"

Selected files found: 5000/5000
Availability check: 2.2 seconds


## Model

Use the exact requested checkpoint and official [OpenCLIP](https://github.com/mlfoundations/open_clip) evaluation preprocessing and tokenizer.

In [50]:
try:
    model, _, preprocess = open_clip.create_model_and_transforms(
        model_name, pretrained=pretrained, precision="fp32", device=device,
        force_quick_gelu=True,
    )
    tokenizer = open_clip.get_tokenizer(model_name)
except Exception as error:
    raise RuntimeError(
        f"STOP: cannot load {model_name}, pretrained={pretrained!r}: {error}"
    ) from error

model.eval()
model.requires_grad_(False)
assert not model.training
assert not any(parameter.requires_grad for parameter in model.parameters())
embedding_dimension = int(model.visual.output_dim)
tokenizer_context_length = int(model.context_length)
assert tokenizer([""]).shape[1] == tokenizer_context_length
print("Model:", model_name, "Checkpoint:", pretrained)
print("Embedding dimension:", embedding_dimension)
print("Tokenizer context length:", tokenizer_context_length)
print("Frozen:", not any(parameter.requires_grad for parameter in model.parameters()))
print("Evaluation preprocessing:", preprocess)

Model: ViT-B-32 Checkpoint: openai
Embedding dimension: 512
Tokenizer context length: 77
Frozen: True
Evaluation preprocessing: Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    MaybeConvertMode()
    MaybeToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)


In [51]:
activation_names = set()
for module in model.modules():
    activation_name = type(module).__name__
    if activation_name in ("GELU", "QuickGELU"):
        activation_names.add(activation_name)
assert activation_names == {"QuickGELU"}, (
    f"STOP: OpenAI checkpoint activation mismatch: {sorted(activation_names)}"
)
print("Activation: QuickGELU; OpenAI checkpoint compatibility checked.")

Activation: QuickGELU; OpenAI checkpoint compatibility checked.


## Image Encoding

In [52]:
evidence_embedding_map = evidence[["id", "image_path", "source"]].copy()
evidence_embedding_map = evidence_embedding_map.rename(columns={"id": "metadata_id"})
evidence_embedding_map.insert(0, "embedding_row", np.arange(len(evidence), dtype=np.int64))
evidence_embedding_map.insert(1, "evidence_row", np.arange(len(evidence), dtype=np.int64))
assert evidence_embedding_map["image_path"].tolist() == evidence["image_path"].tolist()

def normalized_cpu_embeddings(features):
    features = features.float()
    assert torch.isfinite(features).all().item(), "STOP: nonfinite model features"
    norms = torch.linalg.vector_norm(features, dim=1, keepdim=True)
    assert torch.isfinite(norms).all().item(), "STOP: nonfinite feature norms"
    assert (norms > 0).all().item(), "STOP: zero feature norm"
    normalized = features / norms
    return normalized.cpu().numpy().astype(np.float32, copy=False)

In [53]:
image_formats = Counter()

def load_image_tensor(image_path):
    image_format = None
    try:
        with Image.open(resolve_image_path(image_path)) as image:
            image_format = image.format
            rgb_image = image.convert("RGB")
            tensor = preprocess(rgb_image)
    except Exception as error:
        raise RuntimeError(
            f"STOP: preprocessing failed for {image_path}; format={image_format!r}; error={error}"
        ) from error
    image_formats[image_format] += 1
    return tensor

In [54]:
evidence_image_embeddings = np.empty((len(evidence), embedding_dimension), dtype=np.float32)
image_encoding_started = time.perf_counter()
image_preprocessing_seconds = 0.0
image_paths_in_order = evidence_embedding_map["image_path"].tolist()
with torch.inference_mode():
    for start in range(0, len(image_paths_in_order), batch_size):
        stop = min(start + batch_size, len(image_paths_in_order))
        preprocessing_started = time.perf_counter()
        image_tensors = []
        for image_path in image_paths_in_order[start:stop]:
            image_tensors.append(load_image_tensor(image_path))
        image_preprocessing_seconds += time.perf_counter() - preprocessing_started
        image_batch = torch.stack(image_tensors).to(device)
        features = model.encode_image(image_batch)
        embeddings = normalized_cpu_embeddings(features)
        assert embeddings.shape == (stop - start, embedding_dimension)
        evidence_image_embeddings[start:stop] = embeddings
        if start == 0 or stop % 512 == 0 or stop == len(image_paths_in_order):
            elapsed = time.perf_counter() - image_encoding_started
            print(f"Images: {stop}/{len(image_paths_in_order)}; elapsed: {elapsed:.1f}s; "
                  f"read/preprocess: {image_preprocessing_seconds:.1f}s", flush=True)
        del image_batch, features, image_tensors

image_encoding_seconds = time.perf_counter() - image_encoding_started
assert sum(image_formats.values()) == 5000
display(pd.Series(image_formats, name="images").to_frame())

Images: 64/5000; elapsed: 0.6s; read/preprocess: 0.3s
Images: 512/5000; elapsed: 4.7s; read/preprocess: 3.0s
Images: 1024/5000; elapsed: 9.5s; read/preprocess: 6.2s
Images: 1536/5000; elapsed: 14.0s; read/preprocess: 9.2s
Images: 2048/5000; elapsed: 18.8s; read/preprocess: 12.4s
Images: 2560/5000; elapsed: 23.8s; read/preprocess: 15.7s
Images: 3072/5000; elapsed: 29.0s; read/preprocess: 19.3s
Images: 3584/5000; elapsed: 34.9s; read/preprocess: 23.6s
Images: 4096/5000; elapsed: 40.5s; read/preprocess: 27.6s
Images: 4608/5000; elapsed: 47.1s; read/preprocess: 32.5s
Images: 5000/5000; elapsed: 52.1s; read/preprocess: 36.3s


,images
JPEG,4981
GIF,8
BMP,3
PNG,8


## Text Encoding

Evidence uses `caption`; queries use `submitted_caption`. Tokenization uses OpenCLIP's normal context handling.

In [55]:
def encode_captions(captions, label):
    texts = captions.tolist()
    result = np.empty((len(texts), embedding_dimension), dtype=np.float32)
    with torch.inference_mode():
        for start in range(0, len(texts), batch_size):
            stop = min(start + batch_size, len(texts))
            tokens = tokenizer(texts[start:stop]).to(device)
            assert tokens.shape == (stop - start, tokenizer_context_length)
            features = model.encode_text(tokens)
            embeddings = normalized_cpu_embeddings(features)
            assert embeddings.shape == (stop - start, embedding_dimension)
            result[start:stop] = embeddings
            if start == 0 or stop % 512 == 0 or stop == len(texts):
                print(f"{label}: {stop}/{len(texts)}", flush=True)
    return result

evidence_text_embeddings = encode_captions(evidence["caption"], "Evidence captions")
query_text_embeddings = encode_captions(queries["submitted_caption"], "Submitted query captions")

Evidence captions: 64/5000
Evidence captions: 512/5000
Evidence captions: 1024/5000
Evidence captions: 1536/5000
Evidence captions: 2048/5000
Evidence captions: 2560/5000
Evidence captions: 3072/5000
Evidence captions: 3584/5000
Evidence captions: 4096/5000
Evidence captions: 4608/5000
Evidence captions: 5000/5000
Submitted query captions: 64/200
Submitted query captions: 200/200


## Query Mapping

In [56]:
image_row_lookup = evidence_embedding_map.set_index("image_path")["embedding_row"]
matched_rows = queries["query_image_path"].map(image_row_lookup)
assert not matched_rows.isna().any(), "STOP: unresolved query image path"
matched_rows = matched_rows.to_numpy(dtype=np.int64)

query_embedding_map = queries[["query_id", "image_id", "query_image_path"]].copy()
query_embedding_map.insert(0, "query_row", np.arange(len(queries), dtype=np.int64))
query_embedding_map["matched_image_embedding_row"] = matched_rows
matched_metadata_ids = evidence_embedding_map.iloc[matched_rows]["metadata_id"].to_numpy()
assert np.array_equal(matched_metadata_ids, queries["image_id"].to_numpy())
query_image_embeddings = evidence_image_embeddings[matched_rows].copy()
assert len(query_embedding_map) == 200
print("Query image paths resolved:", len(matched_rows), "/", len(queries))
print("Query images selected from existing image embeddings; no second encoding.")

Query image paths resolved: 200 / 200
Query images selected from existing image embeddings; no second encoding.


## Validation

In [57]:
def validate_embeddings(name, matrix, expected_rows):
    assert matrix.shape == (expected_rows, embedding_dimension), f"STOP: {name} shape mismatch"
    assert matrix.dtype == np.float32, f"STOP: {name} must be float32"
    nan_count = int(np.isnan(matrix).sum())
    inf_count = int(np.isinf(matrix).sum())
    assert nan_count == 0 and inf_count == 0, f"STOP: {name} has NaNs or Infs"
    norms = np.linalg.norm(matrix, axis=1)
    assert np.allclose(norms, 1.0, atol=1e-5, rtol=0), f"STOP: {name} norms differ from 1"
    return {
        "matrix": name, "shape": list(matrix.shape), "dtype": str(matrix.dtype),
        "nan_count": nan_count, "inf_count": inf_count,
        "min_l2_norm": float(norms.min()), "mean_l2_norm": float(norms.mean()),
        "max_l2_norm": float(norms.max()),
    }

embedding_arrays = {
    "evidence_image_embeddings.npy": evidence_image_embeddings,
    "query_image_embeddings.npy": query_image_embeddings,
    "evidence_text_embeddings.npy": evidence_text_embeddings,
    "query_text_embeddings.npy": query_text_embeddings,
}
expected_embedding_rows = {
    "evidence_image_embeddings.npy": 5000,
    "query_image_embeddings.npy": 200,
    "evidence_text_embeddings.npy": 5000,
    "query_text_embeddings.npy": 200,
}
validation_rows = []
for filename, matrix in embedding_arrays.items():
    validation_rows.append(validate_embeddings(filename, matrix, expected_embedding_rows[filename]))
display(pd.DataFrame(validation_rows).round(8))

,matrix,shape,dtype,nan_count,inf_count,min_l2_norm,mean_l2_norm,max_l2_norm
0,evidence_image_embeddings.npy,"[5000, 512]",float32,0,0,1.0,1.0,1.0
1,query_image_embeddings.npy,"[200, 512]",float32,0,0,1.0,1.0,1.0
2,evidence_text_embeddings.npy,"[5000, 512]",float32,0,0,1.0,1.0,1.0
3,query_text_embeddings.npy,"[200, 512]",float32,0,0,1.0,1.0,1.0


In [58]:
def validate_alignment(image_map, query_map, arrays):
    assert len(image_map) == len(evidence) == 5000
    assert len(query_map) == len(queries) == 200
    assert image_map["embedding_row"].tolist() == list(range(len(evidence)))
    assert image_map["evidence_row"].tolist() == list(range(len(evidence)))
    assert image_map["metadata_id"].tolist() == evidence["id"].tolist()
    assert image_map["image_path"].tolist() == evidence["image_path"].tolist()
    assert image_map["source"].tolist() == evidence["source"].tolist()
    assert query_map["query_row"].tolist() == list(range(len(queries)))
    assert query_map["query_id"].tolist() == queries["query_id"].tolist()
    assert query_map["image_id"].tolist() == queries["image_id"].tolist()
    assert query_map["query_image_path"].tolist() == queries["query_image_path"].tolist()
    rows = query_map["matched_image_embedding_row"].to_numpy(dtype=np.int64)
    matched_images = image_map.iloc[rows]
    assert matched_images["image_path"].tolist() == queries["query_image_path"].tolist()
    assert matched_images["metadata_id"].tolist() == queries["image_id"].tolist()
    assert np.array_equal(arrays["query_image_embeddings.npy"], arrays["evidence_image_embeddings.npy"][rows])

validate_alignment(evidence_embedding_map, query_embedding_map, embedding_arrays)
print("Full image/text row alignment and query lookup validated.")

Full image/text row alignment and query lookup validated.


In [59]:
sampler = np.random.default_rng(diagnostic_seed)
evidence_sample_rows = np.sort(sampler.choice(len(evidence), size=3, replace=False))
query_sample_rows = np.sort(sampler.choice(len(queries), size=3, replace=False))
evidence_examples = evidence_embedding_map.iloc[evidence_sample_rows].copy()
evidence_examples["evidence_caption"] = evidence.iloc[evidence_sample_rows]["caption"].to_numpy()
evidence_examples["image_norm"] = np.linalg.norm(evidence_image_embeddings[evidence_sample_rows], axis=1)
evidence_examples["text_norm"] = np.linalg.norm(evidence_text_embeddings[evidence_sample_rows], axis=1)
query_examples = query_embedding_map.iloc[query_sample_rows].copy()
query_examples["image_norm"] = np.linalg.norm(query_image_embeddings[query_sample_rows], axis=1)
query_examples["text_norm"] = np.linalg.norm(query_text_embeddings[query_sample_rows], axis=1)
with pd.option_context("display.max_colwidth", 100):
    display(evidence_examples)
    display(query_examples)

,embedding_row,evidence_row,metadata_id,image_path,source,evidence_caption,image_norm,text_norm
446,446,446,97184,visual_news/origin/washington_post/images/0077/673.jpg,washington_post,No blanket ban on Russians for Rio Olympics,1.0,1.0
3272,3272,3272,950802,visual_news/origin/guardian/images/0165/400.jpg,guardian,Moira Young who has won the Costa children s book prize for Blood Red Road,1.0,1.0
3869,3869,3869,1463691,visual_news/origin/bbc/images/0038/801.jpg,bbc,Mr Gulen avoided any rhetoric that could further inflame Turkish tensions,1.0,1.0


,query_row,query_id,image_id,query_image_path,matched_image_embedding_row,image_norm,text_norm
17,17,Q0018,177510,visual_news/origin/washington_post/images/0549/567.jpg,1057,1.0,1.0
139,139,Q0140,606931,visual_news/origin/guardian/images/0310/029.jpg,2750,1.0,1.0
170,170,Q0171,486461,visual_news/origin/guardian/images/0843/600.jpg,2570,1.0,1.0


In [60]:
duplicate_paths = [
    "visual_news/origin/usa_today/images/0316/125.jpg",
    "visual_news/origin/usa_today/images/0627/512.jpg",
]
assert all(image_path in image_row_lookup.index for image_path in duplicate_paths)
duplicate_rows = image_row_lookup.loc[duplicate_paths].to_numpy(dtype=np.int64)
duplicate_pair_cosine = float(np.dot(
    evidence_image_embeddings[duplicate_rows[0]],
    evidence_image_embeddings[duplicate_rows[1]],
))
print("Known duplicate image rows:", duplicate_rows.tolist())
print(f"Duplicate-pair cosine: {duplicate_pair_cosine:.9f}")
assert np.isclose(duplicate_pair_cosine, 1.0, atol=1e-5, rtol=0), "STOP: duplicate-image sanity check failed"

own_caption_cosines = np.sum(
    evidence_image_embeddings[evidence_sample_rows] * evidence_text_embeddings[evidence_sample_rows],
    axis=1,
)
display(pd.DataFrame({
    "evidence_row": evidence_sample_rows,
    "metadata_id": evidence.iloc[evidence_sample_rows]["id"].to_numpy(),
    "own_caption_cosine": own_caption_cosines,
}))
print("Own-caption cosine is a pipeline sanity check, not a factuality or retrieval metric.")

Known duplicate image rows: [2025, 1540]
Duplicate-pair cosine: 1.000000000


,evidence_row,metadata_id,own_caption_cosine
0,446,97184,0.302029
1,3272,950802,0.301468
2,3869,1463691,0.265025


Own-caption cosine is a pipeline sanity check, not a factuality or retrieval metric.


## Save

Write each artifact through a temporary file in the same Drive directory, then reload and verify it.

In [61]:
def temporary_output_path(final_path):
    with tempfile.NamedTemporaryFile(
        dir=final_path.parent, prefix=f".{final_path.name}.", suffix=".tmp", delete=False,
    ) as temporary:
        return Path(temporary.name)

def save_array(filename, matrix):
    final_path = embedding_root / filename
    temporary_path = temporary_output_path(final_path)
    try:
        with temporary_path.open("wb") as handle:
            np.save(handle, matrix, allow_pickle=False)
        temporary_matrix = np.load(temporary_path, allow_pickle=False)
        validate_embeddings(filename, temporary_matrix, expected_embedding_rows[filename])
        assert temporary_matrix.dtype == matrix.dtype
        assert np.array_equal(temporary_matrix, matrix)
        temporary_path.replace(final_path)
        reloaded = np.load(final_path, allow_pickle=False)
        validate_embeddings(filename, reloaded, expected_embedding_rows[filename])
        assert np.array_equal(reloaded, matrix)
        return reloaded
    finally:
        temporary_path.unlink(missing_ok=True)

In [62]:
def save_mapping(filename, table):
    final_path = embedding_root / filename
    temporary_path = temporary_output_path(final_path)
    try:
        table.to_parquet(temporary_path, engine="pyarrow", index=False)
        pd.testing.assert_frame_equal(pd.read_parquet(temporary_path), table)
        temporary_path.replace(final_path)
        reloaded = pd.read_parquet(final_path)
        pd.testing.assert_frame_equal(reloaded, table)
        return reloaded
    finally:
        temporary_path.unlink(missing_ok=True)

saved_arrays = {}
for filename, matrix in embedding_arrays.items():
    saved_arrays[filename] = save_array(filename, matrix)
    print("Saved and reloaded:", filename)
saved_evidence_map = save_mapping("evidence_embedding_map.parquet", evidence_embedding_map)
saved_query_map = save_mapping("query_embedding_map.parquet", query_embedding_map)
validate_alignment(saved_evidence_map, saved_query_map, saved_arrays)

Saved and reloaded: evidence_image_embeddings.npy
Saved and reloaded: query_image_embeddings.npy
Saved and reloaded: evidence_text_embeddings.npy
Saved and reloaded: query_text_embeddings.npy


In [63]:
embedding_config = {
    **environment,
    "model_name": model_name,
    "pretrained": pretrained,
    "activation": "QuickGELU",
    "force_quick_gelu": True,
    "batch_size": batch_size,
    "image_count": len(evidence_embedding_map),
    "query_count": len(queries),
    "evidence_caption_count": len(evidence),
    "query_caption_count": len(queries),
    "embedding_dimension": embedding_dimension,
    "saved_dtype": "float32",
    "frozen": True,
    "image_preprocessing_source": "open_clip.create_model_and_transforms evaluation transform",
    "image_preprocessing": repr(preprocess),
    "image_loading": "Pillow default frame, RGB conversion in memory",
    "tokenizer_source": "open_clip.get_tokenizer(model_name)",
    "tokenizer_context_length": tokenizer_context_length,
    "normalization_method": "float32 L2 normalization per row before CPU numpy conversion",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "diagnostic_seed": diagnostic_seed,
    "image_embedding_order": "pilot_evidence row order, explicit embedding_row and evidence_row",
    "query_embedding_order": "pilot_queries row order",
    "manifest_sha256": manifest_hashes,
    "image_formats": dict(image_formats),
    "image_encoding_seconds": image_encoding_seconds,
    "image_preprocessing_seconds": image_preprocessing_seconds,
    "availability_check_seconds": availability_seconds,
    "duplicate_pair_cosine": duplicate_pair_cosine,
    "validation": validation_rows,
}

In [64]:
def save_config(config):
    final_path = embedding_root / "clip_embedding_config.json"
    temporary_path = temporary_output_path(final_path)
    try:
        temporary_path.write_text(json.dumps(config, indent=2, allow_nan=False), encoding="utf-8")
        assert json.loads(temporary_path.read_text(encoding="utf-8")) == config
        temporary_path.replace(final_path)
        reloaded = json.loads(final_path.read_text(encoding="utf-8"))
        assert reloaded == config
        return reloaded
    finally:
        temporary_path.unlink(missing_ok=True)

for name, manifest_file in manifest_files.items():
    current_hash = hashlib.sha256(manifest_file.read_bytes()).hexdigest()
    assert current_hash == manifest_hashes[name], f"STOP: frozen {name} manifest changed"
assert not model.training
assert not any(parameter.requires_grad for parameter in model.parameters())
saved_config = save_config(embedding_config)

In [65]:
reloaded_validation = []
for filename, matrix in saved_arrays.items():
    reloaded_validation.append(validate_embeddings(filename, matrix, expected_embedding_rows[filename]))
display(pd.DataFrame(reloaded_validation).round(8))
output_files = list(embedding_arrays) + [
    "evidence_embedding_map.parquet", "query_embedding_map.parquet", "clip_embedding_config.json",
]
file_rows = []
for filename in output_files:
    output_file = embedding_root / filename
    assert output_file.is_file()
    file_rows.append({"file": filename, "bytes": output_file.stat().st_size})
display(pd.DataFrame(file_rows))
print("All seven artifacts saved, reloaded, and validated.")
print("No image files or frozen manifests were rewritten.")
print("Embedding stage complete. Stop here for review before FAISS.")

,matrix,shape,dtype,nan_count,inf_count,min_l2_norm,mean_l2_norm,max_l2_norm
0,evidence_image_embeddings.npy,"[5000, 512]",float32,0,0,1.0,1.0,1.0
1,query_image_embeddings.npy,"[200, 512]",float32,0,0,1.0,1.0,1.0
2,evidence_text_embeddings.npy,"[5000, 512]",float32,0,0,1.0,1.0,1.0
3,query_text_embeddings.npy,"[200, 512]",float32,0,0,1.0,1.0,1.0


,file,bytes
0,evidence_image_embeddings.npy,10240128
1,query_image_embeddings.npy,409728
2,evidence_text_embeddings.npy,10240128
3,query_text_embeddings.npy,409728
4,evidence_embedding_map.parquet,137319
5,query_embedding_map.parquet,10371
6,clip_embedding_config.json,3378


All seven artifacts saved, reloaded, and validated.
No image files or frozen manifests were rewritten.
Embedding stage complete. Stop here for review before FAISS.


## Notes

All query image embeddings reuse evidence image rows. Both duplicate records remain separate.
IDs, sources, and labels are bookkeeping only. Similarity does not establish factual correctness.
No FAISS, retrieval, training, annotation, or plots are included.